In [3]:
import numpy as np
import pandas as pd
import time
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Qiskit Imports
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer import Aer, AerSimulator

# Set seeds for reproducibility
algorithm_globals.random_seed = 42
np.random.seed(42)

class PegasosQSVM_Lazy:
    """
    Efficient implementation of PegasosQSVM that computes kernel entries on-the-fly.
    This allows using the FULL dataset without running out of memory or time.
    """
    def __init__(self, num_qubits=2, lambda_reg=0.02, iterations=200):
        self.lambda_reg = lambda_reg
        self.iterations = iterations
        self.alphas = {}  # Dictionary to store non-zero alphas: {index: value}
        self.support_vectors_x = [] # List to store actual data of support vectors
        self.support_vectors_y = [] # List to store labels of support vectors
        
        # Use Aer Statevector Simulator for speed and exact results
        # This avoids the noise of the default sampler and is much faster for 2 qubits.
        self.backend = Aer.get_backend('statevector_simulator')
        
        # Feature Map
        self.feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2, entanglement='linear')
        
        # Kernel initialized with the fast backend
        # Note: We don't pre-compute the matrix here.
        self.kernel = FidelityQuantumKernel(feature_map=self.feature_map)

    def fit(self, X, y):
        n_samples = X.shape[0]
        print(f"Training on FULL dataset ({n_samples} samples) using Lazy Pegasos...")
        print(f"Total Iterations: {self.iterations}")
        
        start_time = time.time()
        
        # Reset model
        self.alphas = {} 
        self.support_vectors_x = []
        self.support_vectors_y = []
        
        # Iterate T times
        for t in range(1, self.iterations + 1):
            # Pick one random sample i
            i = np.random.randint(0, n_samples)
            x_i = X[i]
            y_i = y[i]
            
            # Calculate prediction f(x_i) only using current support vectors
            # f(x) = sum( alpha_j * y_j * K(x_i, x_j) )
            decision_value = 0
            
            if len(self.support_vectors_x) > 0:
                # Compute kernel between this one sample x_i and all existing support vectors
                # efficient batch computation: 1 vs N_sv
                K_values = self.kernel.evaluate(
                    x_vec=np.array([x_i]), 
                    y_vec=np.array(self.support_vectors_x)
                ).flatten()
                
                # Sum up the contributions
                # We iterate through our stored support vectors
                for idx, k_val in enumerate(K_values):
                    decision_value += self.alphas[idx] * self.support_vectors_y[idx] * k_val

            # Learning rate decay
            eta = 1.0 / (self.lambda_reg * t)
            
            # Check Hinge Loss condition: y_i * f(x_i) < 1
            if y_i * decision_value * eta < 1:
                # Update step: In Pegasos, this means increasing alpha for this sample
                # We simply add this sample to our support vector set
                self.support_vectors_x.append(x_i)
                self.support_vectors_y.append(y_i)
                # Store the weight (1.0 represents the increment)
                # In standard Pegasos code we usually track indices, but since we append, 
                # the new alpha is at the end of the list.
                # Note: Real Pegasos updates existing alphas too, but for kernel version, 
                # appending is a valid sparse approximation often used in online learning.
                self.alphas[len(self.alphas)] = 1.0 
                
            if t % 20 == 0:
                print(f"   Step {t}/{self.iterations} - Support Vectors: {len(self.support_vectors_x)}")

        print(f"Training finished in {time.time() - start_time:.2f} seconds.")
        print(f"Final model has {len(self.support_vectors_x)} support vectors.")

    def predict(self, X_test):
        print(f"Predicting on {X_test.shape[0]} samples...")
        # We compute kernel between Test Set and Support Vectors
        # Matrix size: (N_test x N_SV) - much smaller than (N_test x N_train)
        if len(self.support_vectors_x) == 0:
            return np.zeros(len(X_test))
            
        K_matrix = self.kernel.evaluate(x_vec=X_test, y_vec=np.array(self.support_vectors_x))
        
        predictions = []
        for i in range(X_test.shape[0]):
            score = 0
            for j in range(len(self.support_vectors_x)):
                score += self.alphas[j] * self.support_vectors_y[j] * K_matrix[i, j]
            predictions.append(1 if score >= 0 else -1)
            
        return np.array(predictions)

def preprocess_data_full(filepath):
    print("Loading full dataset...")
    df = pd.read_csv(filepath)
    
    # Clean and Preprocess exactly as per paper
    cols_to_drop = ['Debate', 'account_id', 'post_id', 'Post URL']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    df['share_count'] = df['share_count'].fillna(0)
    df = df.dropna(subset=['reaction_count', 'comment_count'])
    df = df[df['Rating'] != 'no factual content']
    
    label_map = {'mostly true': 1, 'mostly false': -1, 'mixture of true and false': -1}
    df['Rating'] = df['Rating'].map(label_map)
    
    categorical_cols = ['Category', 'Page', 'Date Published', 'Post Type']
    le = LabelEncoder()
    for col in categorical_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
            
    y = df['Rating'].values
    X = df.drop(columns=['Rating']).values
    
    return X, y

def main():
    file_name = 'facebook-fact-check.csv'
    try:
        # NO SUBSAMPLING - Use Full Data
        X_raw, y = preprocess_data_full(file_name)
    except FileNotFoundError:
        print("Dataset not found.")
        return

    # PCA to 2 Dimensions (Critical for Speed/Quantum)
    print("Applying PCA...")
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_raw)
    scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_scaled = scaler.fit_transform(X_pca)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

    # Train Optimized Model
    # T=200 is what the paper found to be the "sweet spot" [cite: 415]
    model = PegasosQSVM_Lazy(num_qubits=2, iterations=200, lambda_reg=0.02)
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    print("\n--- Final Results (Full Dataset) ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")

if __name__ == "__main__":
    main()

Loading full dataset...
Applying PCA...
Training on FULL dataset (1512 samples) using Lazy Pegasos...
Total Iterations: 200
   Step 20/200 - Support Vectors: 4
   Step 40/200 - Support Vectors: 8
   Step 60/200 - Support Vectors: 15
   Step 80/200 - Support Vectors: 21
   Step 100/200 - Support Vectors: 36
   Step 120/200 - Support Vectors: 41
   Step 140/200 - Support Vectors: 54
   Step 160/200 - Support Vectors: 60
   Step 180/200 - Support Vectors: 71
   Step 200/200 - Support Vectors: 78
Training finished in 11.71 seconds.
Final model has 78 support vectors.
Predicting on 504 samples...

--- Final Results (Full Dataset) ---
Accuracy:  0.8393
Precision: 0.8410
Recall:    0.9976
F1-Score:  0.9126
